In [2]:
## Now let analyse the sentiment of the headline and assign a score for it using finbert.
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline

model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name)

classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)

headline = "Apple reports record quarterly earnings."

result = classifier(headline)

print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'positive', 'score': 0.9131031036376953}]


In [4]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Venky-MIT/capstone/Filtered_external.csv")
df.head()

,Unnamed: 0,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
0,6680,2020-06-10 07:33:26+00:00,Tech Stocks And FAANGS Strong Again To Start D...,AAPL,https://www.benzinga.com/government/20/06/1622...,JJ Kinahan,NaN,NaN,NaN,NaN,NaN,NaN
1,6681,2020-06-10 04:14:08+00:00,10 Biggest Price Target Changes For Wednesday,AAPL,https://www.benzinga.com/analyst-ratings/price...,Lisa Levin,NaN,NaN,NaN,NaN,NaN,NaN
2,6682,2020-06-10 03:53:47+00:00,"Benzinga Pro's Top 5 Stocks To Watch For Wed.,...",AAPL,https://www.benzinga.com/short-sellers/20/06/1...,Benzinga Newsdesk,NaN,NaN,NaN,NaN,NaN,NaN
3,6683,2020-06-10 03:19:25+00:00,"Deutsche Bank Maintains Buy on Apple, Raises P...",AAPL,https://www.benzinga.com/news/20/06/16219873/d...,Benzinga Newsdesk,NaN,NaN,NaN,NaN,NaN,NaN
4,6684,2020-06-10 02:27:11+00:00,Apple To Let Users Trade In Their Mac Computer...,AAPL,https://www.benzinga.com/news/20/06/16218697/a...,Neer Varshney,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.describe()

,Unnamed: 0,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
count,7.526000e+03,0.0,0.0,0.0,0.0,0.0,0.0
mean,8.996184e+05,NaN,NaN,NaN,NaN,NaN,NaN
std,4.790110e+05,NaN,NaN,NaN,NaN,NaN,NaN
min,6.680000e+03,NaN,NaN,NaN,NaN,NaN,NaN
25%,5.677412e+05,NaN,NaN,NaN,NaN,NaN,NaN
50%,9.257355e+05,NaN,NaN,NaN,NaN,NaN,NaN
75%,1.255421e+06,NaN,NaN,NaN,NaN,NaN,NaN
max,2.923509e+06,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df.Stock_symbol.value_counts()

,count
Stock_symbol,
NVDA,3146
TSLA,1875
GOOGL,1754
AAPL,473
AMZN,278


In [ ]:
def get_sentiment(text):
    if pd.isna(text):
        return None, None
    result = classifier(text)[0]
    return result['label'], result['score']

# Apply sentiment analysis to the 'Article_title' column
df[['sentiment_label', 'sentiment_score']] = df['Article_title'].apply(lambda x: pd.Series(get_sentiment(x)))

#df.to_csv("/content/drive/MyDrive/Venky-MIT/capstone/Filtered_external_sentiment.csv", index=False)
# Display the DataFrame with the new sentiment columns
display(df.head())

,Unnamed: 0,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary,sentiment_label,sentiment_score
0,6680,2020-06-10 07:33:26+00:00,Tech Stocks And FAANGS Strong Again To Start D...,AAPL,https://www.benzinga.com/government/20/06/1622...,JJ Kinahan,NaN,NaN,NaN,NaN,NaN,NaN,positive,0.865888
1,6681,2020-06-10 04:14:08+00:00,10 Biggest Price Target Changes For Wednesday,AAPL,https://www.benzinga.com/analyst-ratings/price...,Lisa Levin,NaN,NaN,NaN,NaN,NaN,NaN,neutral,0.814269
2,6682,2020-06-10 03:53:47+00:00,"Benzinga Pro's Top 5 Stocks To Watch For Wed.,...",AAPL,https://www.benzinga.com/short-sellers/20/06/1...,Benzinga Newsdesk,NaN,NaN,NaN,NaN,NaN,NaN,neutral,0.933910
3,6683,2020-06-10 03:19:25+00:00,"Deutsche Bank Maintains Buy on Apple, Raises P...",AAPL,https://www.benzinga.com/news/20/06/16219873/d...,Benzinga Newsdesk,NaN,NaN,NaN,NaN,NaN,NaN,positive,0.748401
4,6684,2020-06-10 02:27:11+00:00,Apple To Let Users Trade In Their Mac Computer...,AAPL,https://www.benzinga.com/news/20/06/16218697/a...,Neer Varshney,NaN,NaN,NaN,NaN,NaN,NaN,neutral,0.943328


In [ ]:
#df.to_csv("/content/drive/MyDrive/Venky-MIT/capstone/Filtered_external_sentiment.csv", index=False)

## Split the sentiment-scored news by stock ticker

Group the scored dataset on `Stock_symbol` and write one CSV per ticker into a `sentiment_by_ticker/` folder.

In [ ]:
import os

base_dir = "/content/drive/MyDrive/Venky-MIT/capstone"
out_dir = os.path.join(base_dir, "sentiment_by_ticker")
os.makedirs(out_dir, exist_ok=True)

# Split the sentiment-scored dataframe by stock ticker
for ticker, group in df.groupby("Stock_symbol"):
    file_path = os.path.join(out_dir, f"{ticker}_sentiment.csv")
    group.to_csv(file_path, index=False)
    print(f"{ticker}: {len(group)} rows -> {file_path}")

print("\nDone splitting sentiment data by ticker.")

In [ ]:
#find the start date and end date for each stock
from pathlib import Path
import  pandas as  pd
directory =  Path("/Users/I839511/Library/CloudStorage/OneDrive-SAPSE/capstone/ai-financial-intelligence-platform/data/processed/sentiment_by_ticker")

for file in directory.iterdir():
    print (file.name)
    df = pd.read_csv(file)
    first_date = df["Date"].min()
    last_date = df["Date"].max()
    symbol = df["Stock_symbol"].unique()
    print(symbol +"****** "+ first_date + " *******" + last_date)








NVDA_sentiment.csv
<ArrowStringArray>
['NVDA****** 2011-03-03 00:00:00+00:00 *******2020-06-10 08:37:10+00:00']
Length: 1, dtype: str
GOOGL_sentiment.csv
<ArrowStringArray>
['GOOGL****** 2018-07-25 00:00:00+00:00 *******2020-06-10 11:25:13+00:00']
Length: 1, dtype: str
AMZN_sentiment.csv
<ArrowStringArray>
['AMZN****** 2020-04-27 00:00:00+00:00 *******2020-06-10 09:18:50+00:00']
Length: 1, dtype: str
AAPL_sentiment.csv
<ArrowStringArray>
['AAPL****** 2020-03-09 00:00:00+00:00 *******2020-06-10 07:33:26+00:00']
Length: 1, dtype: str
TSLA_sentiment.csv
<ArrowStringArray>
['TSLA****** 2019-07-01 00:00:00+00:00 *******2020-06-10 13:02:47+00:00']
Length: 1, dtype: str


In [19]:
df.head()

,Unnamed: 0,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary,sentiment_label,sentiment_score
0,1255221,2020-06-10 13:02:47+00:00,Tesla's Stock Closes At All-Time High As Musk ...,TSLA,https://www.benzinga.com/news/20/06/16225150/t...,Drew Levine,NaN,NaN,NaN,NaN,NaN,NaN,positive,0.663160
1,1255222,2020-06-10 11:08:09+00:00,'Tesla factory workplace safety is 5% better t...,TSLA,https://www.benzinga.com/news/20/06/16225621/t...,Benzinga Newsdesk,NaN,NaN,NaN,NaN,NaN,NaN,positive,0.946787
2,1255223,2020-06-10 08:41:58+00:00,'Tesla hacker unlocks Performance upgrade and ...,TSLA,https://www.benzinga.com/news/20/06/16224205/t...,Benzinga Newsdesk,NaN,NaN,NaN,NaN,NaN,NaN,positive,0.885173
3,1255224,2020-06-10 07:33:18+00:00,GM On Track To Spend $20B On EV And AV Develop...,TSLA,https://www.benzinga.com/news/20/06/16223414/g...,Benzinga Newsdesk,NaN,NaN,NaN,NaN,NaN,NaN,positive,0.594764
4,1255225,2020-06-10 06:15:07+00:00,"Tesla's Journey To $1,000 In 2020",TSLA,https://www.benzinga.com/news/20/06/16222035/t...,Wayne Duggan,NaN,NaN,NaN,NaN,NaN,NaN,neutral,0.896534


In [22]:

mapping = {
    "positive": 1,
    "neutral": 0,
    "negative": -1
}

df["sentiment"] = df["sentiment_label"].map(mapping)

daily = (
    df
    .groupby(["Date", "Stock_symbol"])
    .agg(
        AvgSentiment=("sentiment", "mean"),
        HeadlineCount=("Article_title", "count"),
        AvgConfidence=("sentiment_score", "mean")
    )
    .reset_index()
)

daily.head()

,Date,Stock_symbol,AvgSentiment,HeadlineCount,AvgConfidence
0,2019-07-01 00:00:00+00:00,TSLA,-0.250000,4,0.870666
1,2019-07-02 00:00:00+00:00,TSLA,0.250000,12,0.877054
2,2019-07-03 00:00:00+00:00,TSLA,0.333333,12,0.790453
3,2019-07-05 00:00:00+00:00,TSLA,0.000000,3,0.858459
4,2019-07-06 00:00:00+00:00,TSLA,0.000000,2,0.939912


In [52]:
def merge_sentiments_for_date(df,file,stock):
    mapping = {
    "positive": 1,
    "neutral": 0,
    "negative": -1
    }

    df["sentiment"] = df["sentiment_label"].map(mapping)

    daily = (
    df
    .groupby(["Date", "Stock_symbol"])
    .agg(
        AvgSentiment=("sentiment", "mean"),
        HeadlineCount=("Article_title", "count"),
        AvgConfidence=("sentiment_score", "mean")
    )
    .reset_index()
    )
    output_file = f"/Users/I839511/Library/CloudStorage/OneDrive-SAPSE/capstone/ai-financial-intelligence-platform/data/processed/daily/{stock}_daily_sentiment.csv"
    daily.to_csv(output_file)
    


In [54]:
from pathlib import Path
import  pandas as  pd
directory =  Path("/Users/I839511/Library/CloudStorage/OneDrive-SAPSE/capstone/ai-financial-intelligence-platform/data/processed/sentiment_by_ticker")

for file in directory.iterdir():
    print (file.name)
    stock= file.stem.split("_")[0]
    df = pd.read_csv(file)
    merge_sentiments_for_date(df,file,stock)



NVDA_sentiment.csv
GOOGL_sentiment.csv
AMZN_sentiment.csv
AAPL_sentiment.csv
TSLA_sentiment.csv
